In [1]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
from google.colab import files

uploaded = files.upload()

Saving train.csv to train.csv


In [2]:
df = pd.read_csv('train.csv')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (9800, 18)
Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales']


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


In [3]:
# Check nulls
print("Missing values:")
print(df.isnull().sum())

Missing values:
Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country           0
City              0
State             0
Postal Code      11
Region            0
Product ID        0
Category          0
Sub-Category      0
Product Name      0
Sales             0
dtype: int64


In [4]:
# Fix date columns - convert to datetime
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)
df['Ship Date'] = pd.to_datetime(df['Ship Date'], dayfirst=True)

# Extract useful date parts
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order Month Name'] = df['Order Date'].dt.strftime('%b')
df['Year Month'] = df['Order Date'].dt.to_period('M').astype(str)

# Calculate days to ship
df['Days to Ship'] = (df['Ship Date'] - df['Order Date']).dt.days

# Clean column names - remove spaces for SQL
df.columns = df.columns.str.replace(' ', '_')

# Round sales
df['Sales'] = df['Sales'].round(2)

# Reset index
df = df.reset_index(drop=True)

print("Cleaned shape:", df.shape)
print("Date range:", df['Order_Date'].min(), "to", df['Order_Date'].max())
df.head()

Cleaned shape: (9800, 23)
Date range: 2015-01-03 00:00:00 to 2018-12-30 00:00:00


,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,...,Product_ID,Category,Sub-Category,Product_Name,Sales,Order_Year,Order_Month,Order_Month_Name,Year_Month,Days_to_Ship
0,1,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2017,11,Nov,2017-11,3
1,2,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,2017,11,Nov,2017-11,3
2,3,CA-2017-138688,2017-06-12,2017-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2017,6,Jun,2017-06,4
3,4,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.58,2016,10,Oct,2016-10,7
4,5,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.37,2016,10,Oct,2016-10,7


In [5]:
conn = sqlite3.connect('superstore.db')

# Load main table
df.to_sql('orders', conn, if_exists='replace', index=False)

# Confirm
result = pd.read_sql_query("SELECT COUNT(*) as total FROM orders", conn)
print("Rows loaded:", result['total'][0])

# Check years
years = pd.read_sql_query("SELECT DISTINCT Order_Year FROM orders ORDER BY Order_Year", conn)
print("Years in data:")
print(years)

Rows loaded: 9800
Years in data:
   Order_Year
0        2015
1        2016
2        2017
3        2018


In [6]:
q1 = pd.read_sql_query("""
    SELECT
        Year_Month,
        Order_Year,
        Order_Month,
        ROUND(SUM(Sales), 2) AS monthly_sales,
        ROUND(SUM(Sales) - LAG(SUM(Sales)) OVER (ORDER BY Year_Month), 2) AS mom_change,
        ROUND(
            (SUM(Sales) - LAG(SUM(Sales)) OVER (ORDER BY Year_Month)) /
            LAG(SUM(Sales)) OVER (ORDER BY Year_Month) * 100, 1
        ) AS mom_pct_change
    FROM orders
    GROUP BY Year_Month, Order_Year, Order_Month
    ORDER BY Year_Month
""", conn)
print(q1)

   Year_Month  Order_Year  Order_Month  monthly_sales  mom_change  \
0     2015-01        2015            1       14205.71         NaN   
1     2015-02        2015            2        4519.92    -9685.79   
2     2015-03        2015            3       55205.81    50685.89   
3     2015-04        2015            4       27906.86   -27298.95   
4     2015-05        2015            5       23644.29    -4262.57   
5     2015-06        2015            6       34322.92    10678.63   
6     2015-07        2015            7       33781.52     -541.40   
7     2015-08        2015            8       27117.53    -6663.99   
8     2015-09        2015            9       81623.50    54505.97   
9     2015-10        2015           10       31453.37   -50170.13   
10    2015-11        2015           11       77907.68    46454.31   
11    2015-12        2015           12       68167.07    -9740.61   
12    2016-01        2016            1       18066.96   -50100.11   
13    2016-02        2016         

In [7]:
q2 = pd.read_sql_query("""
    SELECT
        Region,
        Category,
        ROUND(SUM(Sales), 2) AS total_sales,
        RANK() OVER (PARTITION BY Category ORDER BY SUM(Sales) DESC) AS rank_within_category
    FROM orders
    GROUP BY Region, Category
    ORDER BY Category, rank_within_category
""", conn)
print(q2)

     Region         Category  total_sales  rank_within_category
0      West        Furniture    245348.26                     1
1      East        Furniture    206461.33                     2
2   Central        Furniture    160317.44                     3
3     South        Furniture    116531.47                     4
4      West  Office Supplies    217466.42                     1
5      East  Office Supplies    199940.90                     2
6   Central  Office Supplies    163590.15                     3
7     South  Office Supplies    124424.72                     4
8      East       Technology    263116.56                     1
9      West       Technology    247404.92                     2
10  Central       Technology    168739.19                     3
11    South       Technology    148195.19                     4


In [8]:
q3 = pd.read_sql_query("""
    SELECT
        Customer_Name,
        Segment,
        Region,
        ROUND(SUM(Sales), 2) AS total_sales,
        ROUND(SUM(SUM(Sales)) OVER (ORDER BY SUM(Sales) DESC), 2) AS cumulative_sales,
        RANK() OVER (ORDER BY SUM(Sales) DESC) AS customer_rank
    FROM orders
    GROUP BY Customer_Name, Segment, Region
    ORDER BY total_sales DESC
    LIMIT 15
""", conn)
print(q3)

         Customer_Name      Segment   Region  total_sales  cumulative_sales  \
0          Sean Miller  Home Office    South     23669.21          23669.21   
1         Tamara Chand    Corporate  Central     18437.14          42106.35   
2         Raymond Buch     Consumer     West     14345.28          56451.63   
3         Tom Ashbrook  Home Office     East     13723.50          70175.13   
4        Adrian Barton     Consumer  Central     12181.60          82356.73   
5         Becky Martin     Consumer  Central     10539.90          92896.63   
6         Hunter Lopez     Consumer     East     10522.55         103419.18   
7         Bill Shonely    Corporate     East     10022.29         113441.47   
8         Sanjit Chand     Consumer  Central      9900.19         123341.66   
9            Greg Tran     Consumer     East      9382.94         132724.60   
10         Seth Vernon     Consumer     East      9216.56         141941.16   
11        Sanjit Engle     Consumer    South      88

In [9]:
q4 = pd.read_sql_query("""
    WITH category_totals AS (
        SELECT
            Category,
            ROUND(SUM(Sales), 2) AS category_total
        FROM orders
        GROUP BY Category
    ),
    subcategory_totals AS (
        SELECT
            Category,
            [Sub-Category],
            ROUND(SUM(Sales), 2) AS subcategory_total,
            COUNT(DISTINCT Order_ID) AS order_count,
            ROUND(AVG(Sales), 2) AS avg_order_value
        FROM orders
        GROUP BY Category, [Sub-Category]
    )
    SELECT
        s.Category,
        s.[Sub-Category],
        s.subcategory_total,
        s.order_count,
        s.avg_order_value,
        c.category_total,
        ROUND(s.subcategory_total / c.category_total * 100, 1) AS pct_of_category
    FROM subcategory_totals s
    JOIN category_totals c ON s.Category = c.Category
    ORDER BY s.Category, s.subcategory_total DESC
""", conn)
print(q4)

           Category Sub-Category  subcategory_total  order_count  \
0         Furniture       Chairs          322822.72          566   
1         Furniture       Tables          202810.62          302   
2         Furniture    Bookcases          113813.18          222   
3         Furniture  Furnishings           89211.98          855   
4   Office Supplies      Storage          219343.37          764   
5   Office Supplies      Binders          200028.73         1291   
6   Office Supplies   Appliances          104618.38          444   
7   Office Supplies        Paper           76828.34         1163   
8   Office Supplies     Supplies           46420.29          181   
9   Office Supplies          Art           26705.42          720   
10  Office Supplies    Envelopes           16128.02          243   
11  Office Supplies       Labels           12347.71          340   
12  Office Supplies    Fasteners            3001.93          212   
13       Technology       Phones          327782

In [10]:
q5 = pd.read_sql_query("""
    WITH shipping_stats AS (
        SELECT
            Segment,
            Ship_Mode,
            COUNT(*) AS order_count,
            ROUND(AVG(Days_to_Ship), 1) AS avg_days_to_ship,
            ROUND(SUM(Sales), 2) AS total_sales,
            ROW_NUMBER() OVER (PARTITION BY Segment ORDER BY COUNT(*) DESC) AS rn
        FROM orders
        GROUP BY Segment, Ship_Mode
    )
    SELECT
        Segment,
        Ship_Mode,
        order_count,
        avg_days_to_ship,
        total_sales
    FROM shipping_stats
    ORDER BY Segment, order_count DESC
""", conn)
print(q5)

        Segment       Ship_Mode  order_count  avg_days_to_ship  total_sales
0      Consumer  Standard Class         3031               5.0    702377.80
1      Consumer    Second Class         1003               3.2    230125.45
2      Consumer     First Class          755               2.2    158104.83
3      Consumer        Same Day          312               0.0     57452.21
4     Corporate  Standard Class         1782               5.0    401747.41
5     Corporate    Second Class          589               3.3    139045.25
6     Corporate     First Class          468               2.2    102580.02
7     Corporate        Same Day          114               0.1     45121.34
8   Home Office  Standard Class         1046               5.0    236706.19
9   Home Office    Second Class          310               3.3     80743.30
10  Home Office     First Class          278               2.1     84887.30
11  Home Office        Same Day          112               0.1     22645.45


In [11]:
df.to_csv('superstore_clean.csv', index=False)
q1.to_csv('monthly_sales_trend.csv', index=False)
q2.to_csv('regional_category_ranking.csv', index=False)
q3.to_csv('top_customers.csv', index=False)
q4.to_csv('subcategory_performance.csv', index=False)
q5.to_csv('shipping_analysis.csv', index=False)
print("All files exported!")

All files exported!


In [12]:
from google.colab import files
files.download('superstore_clean.csv')
files.download('monthly_sales_trend.csv')
files.download('regional_category_ranking.csv')
files.download('top_customers.csv')
files.download('subcategory_performance.csv')
files.download('shipping_analysis.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
kpis = {
    'Total Sales': f"${df['Sales'].sum():,.0f}",
    'Total Orders': f"{df['Order_ID'].nunique():,}",
    'Avg Order Value': f"${df.groupby('Order_ID')['Sales'].sum().mean():,.0f}",
    'Total Customers': f"{df['Customer_ID'].nunique():,}",
    'Top Region': df.groupby('Region')['Sales'].sum().idxmax()
}
for k, v in kpis.items():
    print(f"{k}: {v}")

Total Sales: $2,261,537
Total Orders: 4,922
Avg Order Value: $459
Total Customers: 793
Top Region: West
